In [5]:
import json
from urllib.request import urlopen
from collections import Counter

# Original dataset from the Pain Axis paper
url = (
    "https://raw.githubusercontent.com/"
    "valen-research/Pain-axis/main/datasets/"
    "3.1_pain_and_control_datasets.json"
)

with urlopen(url) as response:
    data = json.load(response)

# Select first-person naturalistic sentences
samples = data["datasets"]["S2_1P"]["sentences"]

print("Total samples:", len(samples))
print("Categories:")
print(Counter(s["category"] for s in samples))

# Display the first five examples
for sample in samples[:5]:
    print(sample["category"], sample["prompt"])

Total samples: 200
Categories:
Counter({'A1': 20, 'A2': 20, 'A3': 20, 'A4': 20, 'A5': 20, 'B': 20, 'C1': 20, 'C2': 20, 'D': 20, 'E': 20})
A1 The knife slices into my finger. I feel:
A1 My tooth throbs with each heartbeat. I feel:
A1 A migraine pounds behind my eyes. I feel:
A1 I twist my ankle on the stairs. I feel:
A1 The hot oil splatters onto my arm. I feel:


In [6]:
import pandas as pd

labels = data["metadata"]["category_labels"]

df = pd.DataFrame(samples)
df["category_name"] = df["category"].map(labels)

display(df[["category", "category_name", "prompt"]].head(10))

print(df["category_name"].value_counts())


,category,category_name,prompt
0,A1,Physical Pain,The knife slices into my finger. I feel:
1,A1,Physical Pain,My tooth throbs with each heartbeat. I feel:
2,A1,Physical Pain,A migraine pounds behind my eyes. I feel:
3,A1,Physical Pain,I twist my ankle on the stairs. I feel:
4,A1,Physical Pain,The hot oil splatters onto my arm. I feel:
5,A1,Physical Pain,My back seizes up when I bend over. I feel:
6,A1,Physical Pain,A bee stings me on the neck. I feel:
7,A1,Physical Pain,I bite down hard on my tongue. I feel:
8,A1,Physical Pain,The sunburn on my shoulders is peeling. I feel:
9,A1,Physical Pain,My knee buckles under my weight. I feel:


category_name
Physical Pain                 20
Psychological Pain            20
Social Pain                   20
Moral Injury                  20
Cognitive Pain                20
Fear                          20
Negative Emotion              20
Negative World State          20
Neutral                       20
Non-painful Body Sensation    20
Name: count, dtype: int64


In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32
).to("cuda")

model.eval()

print("Model loaded successfully!")
print("Layers:", model.config.num_hidden_layers)
print("Hidden size:", model.config.hidden_size)


Model loaded successfully!
Layers: 24
Hidden size: 896


In [8]:
pain_text = next(
    s["prompt"] for s in samples if s["category"] == "A1"
)

neutral_text = next(
    s["prompt"] for s in samples if s["category"] == "D"
)

def extract_activation(text, layer=10):
    inputs = tokenizer(
        text, return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model(
            **inputs,
            output_hidden_states=True
        )

    return outputs.hidden_states[layer][0, -1, :].cpu()

pain_activation = extract_activation(pain_text)
neutral_activation = extract_activation(neutral_text)

print("Pain:", pain_activation.shape)
print("Neutral:", neutral_activation.shape)


We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


Pain: torch.Size([896])
Neutral: torch.Size([896])


In [10]:
import torch

pain_categories = ["A1", "A2", "A3", "A4", "A5"]
control_categories = ["B", "C1", "C2", "D", "E"]

balanced = df.groupby(
    "category", group_keys=False
).sample(n=2, random_state=42)

pain_samples = balanced[
    balanced["category"].isin(pain_categories)
]["prompt"].tolist()

control_samples = balanced[
    balanced["category"].isin(control_categories)
]["prompt"].tolist()

print("Pain samples:", len(pain_samples))
print("Control samples:", len(control_samples))

pain_activations = torch.stack([
    extract_activation(text) for text in pain_samples
])

control_activations = torch.stack([
    extract_activation(text) for text in control_samples
])

pain_direction = (
    pain_activations.mean(dim=0)
    - control_activations.mean(dim=0)
)

print("Control activations:", control_activations.shape)
print("Pain direction:", pain_direction.shape)
print("Direction norm:", torch.norm(pain_direction).item())


Pain samples: 10
Control samples: 10
Control activations: torch.Size([10, 896])
Pain direction: torch.Size([896])
Direction norm: 1.8282687664031982
